# Unemployment rate — incremental Silver load

**Source:** `pondolens.dev_bronze.unemployment_rate`  
**Target:** `pondolens.dev_silver.unemployment_rate`  
**Business key:** `(year, region_code)`  
**Surrogate ID:** INT, starting at 1000001, allocated only for new keys. Existing IDs are never reassigned.

Run sequentially in a **Databricks SQL** notebook, in a single-writer scheduled job. Bronze remains unchanged. The source is re-evaluated every run; this is an idempotent snapshot/upsert workflow, not a CDC delete pipeline. A repeated run with unchanged data makes no Silver updates. New data is inserted; changed rates/names are updated. Source selection prioritizes final estimates then latest release.

**Important:** This job requires exclusive writes to the target while it allocates IDs from `MAX(unemployment_rate_id)`. Concurrent runs can allocate overlapping IDs. Disable overlapping job runs, and avoid other writers. Existing tables must have unique, non-null business keys and valid INT IDs. The notebook does not reset or overwrite existing IDs.


## 1. Create the target only if it does not exist

In [ ]:
CREATE SCHEMA IF NOT EXISTS pondolens.dev_silver;

CREATE TABLE IF NOT EXISTS pondolens.dev_silver.unemployment_rate (
  unemployment_rate_id INT,
  year INT,
  region_code STRING,
  region_name STRING,
  unemployment_rate DECIMAL(12,4),
  run_id STRING,
  source STRING,
  processed_at TIMESTAMP
) USING DELTA;

## 2. Prepare the cleaned incoming batch

This temporary view applies the existing region mapping, year conversion (`2021.0` → `2021`), rate validation and deterministic estimate preference. Review rejected/unmapped records separately.

In [ ]:
CREATE OR REPLACE TEMP VIEW unemployment_incoming AS
WITH normalized AS (
 SELECT TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) AS year,
        UPPER(TRIM(REGEXP_REPLACE(COALESCE(NULLIF(TRIM(region), ''), geographic_name), '\s+', ' '))) AS raw_region_name,
        TRY_CAST(REGEXP_REPLACE(TRIM(CAST(value AS STRING)), ',', '') AS DECIMAL(12,4)) AS unemployment_rate,
        LOWER(TRIM(estimate_status)) AS estimate_status,
        COALESCE(table_version, '') AS table_version
 FROM pondolens.dev_bronze.unemployment_rate
 WHERE TRY_CAST(TRY_CAST(TRIM(year) AS DECIMAL(10,1)) AS INT) BETWEEN 2021 AND 2025
   AND LOWER(TRIM(geographic_level)) = 'region'
   AND LOWER(TRIM(indicator)) = 'unemployment rate'
), mapped AS (
 SELECT *,
   CASE
     WHEN raw_region_name RLIKE 'NATIONAL CAPITAL REGION|\(NCR\)' THEN 'NCR'
     WHEN raw_region_name RLIKE 'CORDILLERA ADMINISTRATIVE REGION|\(CAR\)' THEN 'CAR'
     WHEN raw_region_name RLIKE 'ILOCOS|REGION I \(' THEN 'I'
     WHEN raw_region_name RLIKE 'CAGAYAN VALLEY|REGION II \(' THEN 'II'
     WHEN raw_region_name RLIKE 'CENTRAL LUZON|REGION III \(' THEN 'III'
     WHEN raw_region_name RLIKE 'CALABARZON|REGION IV-A|REGION IVA' THEN 'IV-A'
     WHEN raw_region_name RLIKE 'MIMAROPA|REGION IV-B|REGION IVB' THEN 'IV-B'
     WHEN raw_region_name RLIKE 'BICOL|REGION V \(' THEN 'V'
     WHEN raw_region_name RLIKE 'WESTERN VISAYAS|REGION VI \(' THEN 'VI'
     WHEN raw_region_name RLIKE 'CENTRAL VISAYAS|REGION VII \(' THEN 'VII'
     WHEN raw_region_name RLIKE 'EASTERN VISAYAS|REGION VIII \(' THEN 'VIII'
     WHEN raw_region_name RLIKE 'ZAMBOANGA PENINSULA|REGION IX \(' THEN 'IX'
     WHEN raw_region_name RLIKE 'NORTHERN MINDANAO|REGION X \(' THEN 'X'
     WHEN raw_region_name RLIKE 'DAVAO|REGION XI \(' THEN 'XI'
     WHEN raw_region_name RLIKE 'SOCCSKSARGEN|REGION XII \(' THEN 'XII'
     WHEN raw_region_name RLIKE 'CARAGA|REGION XIII \(' THEN 'XIII'
     WHEN raw_region_name RLIKE 'BANGSAMORO|BARMM|ARMM|AUTONOMOUS REGION IN MUSLIM MINDANAO' THEN 'BARMM'
     WHEN raw_region_name RLIKE 'NEGROS ISLAND REGION|\(NIR\)' THEN 'NIR'
   END AS region_code
 FROM normalized
), valid AS (
 SELECT *,
 CASE region_code
 WHEN 'NCR' THEN 'National Capital Region' WHEN 'CAR' THEN 'Cordillera Administrative Region'
 WHEN 'I' THEN 'Ilocos Region' WHEN 'II' THEN 'Cagayan Valley' WHEN 'III' THEN 'Central Luzon'
 WHEN 'IV-A' THEN 'CALABARZON' WHEN 'IV-B' THEN 'MIMAROPA' WHEN 'V' THEN 'Bicol Region'
 WHEN 'VI' THEN 'Western Visayas' WHEN 'VII' THEN 'Central Visayas' WHEN 'VIII' THEN 'Eastern Visayas'
 WHEN 'IX' THEN 'Zamboanga Peninsula' WHEN 'X' THEN 'Northern Mindanao' WHEN 'XI' THEN 'Davao Region'
 WHEN 'XII' THEN 'SOCCSKSARGEN' WHEN 'XIII' THEN 'Caraga' WHEN 'BARMM' THEN 'BARMM'
 WHEN 'NIR' THEN 'Negros Island Region' END AS region_name,
 COALESCE(TRY_CAST(REGEXP_EXTRACT(table_version, '(20[0-9]{2})(?!.*20[0-9]{2})', 1) AS INT), 0) AS release_year
 FROM mapped WHERE region_code IS NOT NULL AND unemployment_rate BETWEEN 0 AND 100
), ranked AS (
 SELECT *, ROW_NUMBER() OVER (PARTITION BY year, region_code
 ORDER BY CASE estimate_status WHEN 'final' THEN 0 WHEN 'preliminary' THEN 1 ELSE 2 END,
 release_year DESC, unemployment_rate DESC, table_version DESC) AS selection_rank
 FROM valid
)
SELECT year, region_code, region_name, unemployment_rate,
       'pondolens.dev_bronze.unemployment_rate' AS source
FROM ranked WHERE selection_rank = 1;

## 3. Validate the incoming batch and existing IDs

**Stop the job** if any of these queries return unexpected results. The later SQL `assert_true` cell fails automatically for empty batches, invalid keys, duplicate keys, invalid target IDs, or insufficient INT capacity.

In [ ]:
SELECT year, COUNT(*) AS regions FROM unemployment_incoming GROUP BY year ORDER BY year;
SELECT year, region_code, COUNT(*) AS n FROM unemployment_incoming GROUP BY year, region_code HAVING COUNT(*) > 1;
SELECT year, region_code, COUNT(*) AS n FROM pondolens.dev_silver.unemployment_rate GROUP BY year, region_code HAVING COUNT(*) > 1;

## 4. Determine new IDs without changing existing records

The staged source assigns IDs only to keys absent from Silver. The target is not replaced. IDs are allocated deterministically in `(year, region_code)` order, starting at `1000001` for an empty target.

In [ ]:
CREATE OR REPLACE TEMP VIEW unemployment_upsert AS
WITH current_max AS (
 SELECT GREATEST(COALESCE(MAX(CAST(unemployment_rate_id AS BIGINT)), 1000000), 1000000) AS max_id
 FROM pondolens.dev_silver.unemployment_rate
), new_keys AS (
 SELECT i.year, i.region_code,
        ROW_NUMBER() OVER (ORDER BY i.year, i.region_code) AS seq
 FROM unemployment_incoming i
 LEFT ANTI JOIN pondolens.dev_silver.unemployment_rate t
   ON i.year = t.year AND i.region_code = t.region_code
), numbered AS (
 SELECT k.year, k.region_code, m.max_id + k.seq AS new_id
 FROM new_keys k CROSS JOIN current_max m
)
SELECT i.*, CAST(n.new_id AS INT) AS new_id
FROM unemployment_incoming i
LEFT JOIN numbered n ON i.year = n.year AND i.region_code = n.region_code;

## 5. Guardrails — fail before writing

Run this cell and resolve any error before the MERGE. The check ensures the largest allocated ID fits a signed INT.

In [ ]:
SELECT assert_true(
 (SELECT COUNT(*) FROM unemployment_incoming) > 0
 AND (SELECT COUNT(*) FROM unemployment_incoming WHERE year IS NULL OR region_code IS NULL OR unemployment_rate IS NULL) = 0
 AND (SELECT COUNT(*) FROM (SELECT year, region_code FROM unemployment_incoming GROUP BY year, region_code HAVING COUNT(*) > 1)) = 0
 AND (SELECT COUNT(*) FROM pondolens.dev_silver.unemployment_rate WHERE year IS NULL OR region_code IS NULL OR unemployment_rate_id IS NULL OR unemployment_rate_id < 1000001) = 0
 AND (SELECT COUNT(*) FROM (SELECT year, region_code FROM pondolens.dev_silver.unemployment_rate GROUP BY year, region_code HAVING COUNT(*) > 1)) = 0
 AND (SELECT COUNT(*) FROM (SELECT unemployment_rate_id FROM pondolens.dev_silver.unemployment_rate GROUP BY unemployment_rate_id HAVING COUNT(*) > 1)) = 0
 AND (SELECT COALESCE(MAX(CAST(new_id AS BIGINT)), 0) FROM unemployment_upsert) <= 2147483647,
 'Batch validation failed: inspect incoming keys, target IDs, empty input, or INT capacity'
) AS validation_result;

## 6. Incremental upsert (MERGE)

Matched records update only when rate, name or source changed. Unchanged records keep their ID, `run_id` and `processed_at`. New records receive their allocated ID.

In [ ]:
MERGE INTO pondolens.dev_silver.unemployment_rate AS t
USING unemployment_upsert AS s
ON t.year = s.year AND t.region_code = s.region_code
WHEN MATCHED AND (
  NOT (t.unemployment_rate <=> s.unemployment_rate)
  OR NOT (t.region_name <=> s.region_name)
  OR NOT (t.source <=> s.source)
) THEN UPDATE SET
  t.region_name = s.region_name,
  t.unemployment_rate = s.unemployment_rate,
  t.source = s.source,
  t.run_id = CONCAT('unemployment_', DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss')),
  t.processed_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
  unemployment_rate_id, year, region_code, region_name,
  unemployment_rate, run_id, source, processed_at
) VALUES (
  s.new_id, s.year, s.region_code, s.region_name,
  s.unemployment_rate,
  CONCAT('unemployment_', DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd_HHmmss')),
  s.source, CURRENT_TIMESTAMP()
);

## 7. Post-load validation

In [ ]:
SELECT COUNT(*) AS silver_rows, MIN(unemployment_rate_id) AS min_id,
 MAX(unemployment_rate_id) AS max_id, COUNT(DISTINCT unemployment_rate_id) AS distinct_ids
FROM pondolens.dev_silver.unemployment_rate;

SELECT year, COUNT(*) AS row_count, COUNT(DISTINCT region_code) AS distinct_regions
FROM pondolens.dev_silver.unemployment_rate GROUP BY year ORDER BY year;

SELECT year, region_code, COUNT(*) AS n FROM pondolens.dev_silver.unemployment_rate
GROUP BY year, region_code HAVING COUNT(*) > 1;

## Operational notes

- **Single writer required.** `MAX(ID) + ROW_NUMBER()` is not a safe global allocator under concurrent writers. Disable overlapping runs and avoid manual writes during a run.
- **Snapshot semantics:** if an older Bronze release remains alongside a newer one, the ranking determines the chosen record. Validate that final-over-preliminary and release-year ordering match your business rules. No deletes are performed.
- **Region mapping:** The source mapping is inherited from the prior notebook. Inspect unmapped Bronze regions and check counts by year; unmatched regions are not inserted.
- **Historical records:** The current target is preserved. If a previous load mis-mapped a region code, that is a business-key correction and needs a controlled migration rather than an ordinary MERGE.
- **Staging:** The view is temporary within the same SQL session. Run all cells in order, without overlapping jobs. The allocation query assumes no concurrent target changes between its validation and MERGE.
- **INT range:** `2147483647` is the maximum signed INT. For very large growth, migrate the ID column to BIGINT in a planned schema change.
